# Object Detection Project

## 1. Settings

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/HoangDuonng1359/object-detection"
PROJECT_DIR = Path("/kaggle/working/object-detection")

IMAGE_SIZE = 512
EPOCHS = 80
BATCH_SIZE = 8
NUM_WORKERS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
USE_PRETRAINED_BACKBONE = True
FREEZE_BACKBONE_STEM = False
OVERSAMPLE_CLASSES = ["chair"]
OVERSAMPLE_FACTOR = 2.0

# Set these to small values for a quick smoke test. Use 0 for full train/validation.
MAX_TRAIN_BATCHES = 0
MAX_VAL_BATCHES = 0

CONF_THRESHOLD = 0.25
NMS_THRESHOLD = 0.35
MAX_DETECTIONS = 100

RUN_THRESHOLD_SWEEP = True
CONF_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
NMS_VALUES = [0.35, 0.45, 0.50, 0.60]

# Optional hidden/test image directory. Leave as None if you only want val predictions.
TEST_IMAGE_DIR = None
FINAL_PREDICTIONS = Path("/kaggle/working/predictions.json")


## 2. Clone Code And Install Requirements

In [2]:
import os
import subprocess
import sys

if not PROJECT_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
        raise ValueError("Edit REPO_URL in the settings cell before cloning.")
    clone_cmd = ["git", "clone"]
    clone_cmd += [REPO_URL, str(PROJECT_DIR)]
    subprocess.run(clone_cmd, check=True)
else:
    print(f"Project directory already exists: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

requirements = PROJECT_DIR / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

import torch
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())


Cloning into '/kaggle/working/object-detection'...


cwd: /kaggle/working/object-detection
python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.10.0+cu128
cuda_available: True


## 3. Locate Kaggle Input Dataset

In [3]:
PUBLIC_DIR = Path("/kaggle/input/datasets/duong1589/object-detection/public")
TRAIN_JSON = PUBLIC_DIR / "annotations" / "train.json"
VAL_JSON = PUBLIC_DIR / "annotations" / "val.json"
TRAIN_IMAGES = PUBLIC_DIR / "train" / "images"
VAL_IMAGES = PUBLIC_DIR / "val" / "images"
EVALUATOR = PUBLIC_DIR / "tools" / "evaluate_predictions.py"

print("PUBLIC_DIR:", PUBLIC_DIR)
print("TRAIN_JSON:", TRAIN_JSON)
print("VAL_JSON:", VAL_JSON)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("VAL_IMAGES:", VAL_IMAGES)
print("EVALUATOR:", EVALUATOR, EVALUATOR.exists())


PUBLIC_DIR: /kaggle/input/datasets/duong1589/object-detection/public
TRAIN_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json
VAL_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json
TRAIN_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/train/images
VAL_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/val/images
EVALUATOR: /kaggle/input/datasets/duong1589/object-detection/public/tools/evaluate_predictions.py True


## 4. Sanity Check Dataset And Code

In [4]:
import json

for split_name, json_path in [("train", TRAIN_JSON), ("val", VAL_JSON)]:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    print(split_name, "images=", len(data["images"]), "annotations=", len(data["annotations"]))

subprocess.run([sys.executable, "-m", "py_compile", "train.py", "predict.py"], check=True)
subprocess.run([
    sys.executable, "-m", "utils.check_dataset",
    "--annotations", str(TRAIN_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--image_size", str(IMAGE_SIZE),
    "--batch_size", "2",
    "--train",
], check=True)


train images= 7500 annotations= 10642
val images= 1500 annotations= 2021
dataset_size=7500
classes=['person', 'car', 'dog', 'cat', 'chair']
sample_image_shape=(3, 512, 512)
sample_image_id=img_000c52c6d12f.jpg
sample_boxes=(1, 4)
batch_image_shape=(2, 3, 512, 512)
batch_box_counts=[1, 1]


CompletedProcess(args=['/usr/bin/python3', '-m', 'utils.check_dataset', '--annotations', '/kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json', '--image_dir', '/kaggle/input/datasets/duong1589/object-detection/public/train/images', '--image_size', '512', '--batch_size', '2', '--train'], returncode=0)

## 5. Train

In [5]:
from IPython.display import display
import pandas as pd

train_cmd = [
    sys.executable, "train.py",
    "--train_data", str(TRAIN_JSON),
    "--val_data", str(VAL_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--val_image_dir", str(VAL_IMAGES),
    "--checkpoint_dir", str(PROJECT_DIR / "models"),
    "--image_size", str(IMAGE_SIZE),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--lr", str(LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--nms_threshold", str(NMS_THRESHOLD),
]
if USE_AMP:
    train_cmd.append("--amp")
if USE_PRETRAINED_BACKBONE:
    train_cmd.append("--pretrained_backbone")
if FREEZE_BACKBONE_STEM:
    train_cmd.append("--freeze_backbone_stem")
if OVERSAMPLE_CLASSES and OVERSAMPLE_FACTOR > 1.0:
    train_cmd += ["--oversample_classes", *OVERSAMPLE_CLASSES]
    train_cmd += ["--oversample_factor", str(OVERSAMPLE_FACTOR)]
if MAX_TRAIN_BATCHES > 0:
    train_cmd += ["--max_train_batches", str(MAX_TRAIN_BATCHES)]
if MAX_VAL_BATCHES > 0:
    train_cmd += ["--max_val_batches", str(MAX_VAL_BATCHES)]

print("Train command:")
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True)

history_files = sorted((PROJECT_DIR / "models").glob("train_history_*.csv"), key=lambda item: item.stat().st_mtime)
if history_files:
    HISTORY_CSV = history_files[-1]
    history_df = pd.read_csv(HISTORY_CSV)
    print("History CSV:", HISTORY_CSV)
    display_columns = [
        "epoch", "lr", "train_loss", "val_loss", "val_map50", "best_map50",
        "train_box_loss", "train_objectness_loss", "train_class_loss",
        "val_box_loss", "val_objectness_loss", "val_class_loss",
    ]
    display(history_df[[col for col in display_columns if col in history_df.columns]].tail(10))
else:
    HISTORY_CSV = None
    print("No train history CSV found.")


Train command:
/usr/bin/python3 train.py --train_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json --val_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json --image_dir /kaggle/input/datasets/duong1589/object-detection/public/train/images --val_image_dir /kaggle/input/datasets/duong1589/object-detection/public/val/images --checkpoint_dir /kaggle/working/object-detection/models --image_size 512 --epochs 80 --batch_size 8 --num_workers 2 --lr 0.0001 --weight_decay 0.0001 --conf_threshold 0.25 --nms_threshold 0.35 --amp --pretrained_backbone --oversample_classes chair --oversample_factor 2.0
device=cuda amp=True
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s]


epoch=1/80 lr=6.66667e-05 train_loss=3.1453 val_loss=2.6510 val_map50=0.3178 best_map50=0.3178


epoch=2/80 lr=0.0001 train_loss=2.5645 val_loss=2.5319 val_map50=0.4269 best_map50=0.4269


epoch=3/80 lr=0.0001 train_loss=2.3917 val_loss=2.3702 val_map50=0.4387 best_map50=0.4387


...

epoch=77/80 lr=3.74075e-07 train_loss=1.0232 val_loss=1.9175 val_map50=0.6187 best_map50=0.6332


epoch=78/80 lr=1.66371e-07 train_loss=1.0184 val_loss=1.9233 val_map50=0.6232 best_map50=0.6332


epoch=79/80 lr=4.161e-08 train_loss=1.0161 val_loss=1.9175 val_map50=0.6266 best_map50=0.6332


epoch=80/80 lr=0 train_loss=1.0183 val_loss=1.9223 val_map50=0.6251 best_map50=0.6332
best_checkpoint=/kaggle/working/object-detection/models/best.pth
history=/kaggle/working/object-detection/models/train_history_20260529_154823.csv
History CSV: /kaggle/working/object-detection/models/train_history_20260529_154823.csv


,epoch,lr,train_loss,val_loss,val_map50,best_map50,train_box_loss,train_objectness_loss,train_class_loss,val_box_loss,val_objectness_loss,val_class_loss
70,71,3.333174e-06,1.020152,1.919731,0.619393,0.633244,0.196427,0.034295,0.007445,0.297976,0.299874,0.259956
71,72,2.639849e-06,1.023358,1.925967,0.623000,0.633244,0.196944,0.034947,0.007381,0.298067,0.305830,0.259608
72,73,2.025351e-06,1.026917,1.917191,0.622879,0.633244,0.197864,0.034237,0.006723,0.297396,0.299860,0.260707
73,74,1.490702e-06,1.026764,1.928222,0.620214,0.633244,0.197610,0.034213,0.008997,0.297733,0.306960,0.265194
74,75,1.036792e-06,1.034313,1.920126,0.625077,0.633244,0.198992,0.035396,0.007911,0.297605,0.302091,0.260018
75,76,6.643763e-07,1.016802,1.913788,0.629648,0.633244,0.195943,0.033225,0.007723,0.297190,0.298006,0.259668
76,77,3.740749e-07,1.023151,1.917499,0.618682,0.633244,0.196920,0.034047,0.009006,0.297576,0.300629,0.257983
77,78,1.663709e-07,1.018444,1.923296,0.623164,0.633244,0.196521,0.032525,0.006627,0.297693,0.302981,0.263696
78,79,4.161003e-08,1.016093,1.917537,0.626622,0.633244,0.195854,0.033591,0.006467,0.297331,0.300502,0.260756
79,80,0.000000e+00,1.018318,1.922301,0.625121,0.633244,0.196095,0.033669,0.008345,0.297798,0.302321,0.261984


## 6. Predict On Validation And Evaluate

In [6]:
VAL_PREDICTIONS = PROJECT_DIR / "val_predictions.json"
predict_val_cmd = [
    sys.executable, "predict.py",
    "--image_dir", str(VAL_IMAGES),
    "--output", str(VAL_PREDICTIONS),
    "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
    "--batch_size", str(BATCH_SIZE),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--nms_threshold", str(NMS_THRESHOLD),
    "--max_detections", str(MAX_DETECTIONS),
]
print("Predict validation command:")
print(" ".join(predict_val_cmd))
subprocess.run(predict_val_cmd, check=True)

val_predictions_data = json.loads(VAL_PREDICTIONS.read_text(encoding="utf-8"))
num_pred_boxes = sum(len(item["boxes"]) for item in val_predictions_data)
print(f"Validation predictions: images={len(val_predictions_data)} boxes={num_pred_boxes}")
print("Prediction preview:")
print(json.dumps(val_predictions_data[:3], ensure_ascii=False, indent=2))

if EVALUATOR.exists():
    VAL_SCORE = PROJECT_DIR / "val_score.json"
    subprocess.run([
        sys.executable, str(EVALUATOR),
        "--ground_truth", str(VAL_JSON),
        "--predictions", str(VAL_PREDICTIONS),
        "--output", str(VAL_SCORE),
    ], check=True)
    score = json.loads(VAL_SCORE.read_text(encoding="utf-8"))
    print("Validation score:")
    print(json.dumps(score, ensure_ascii=False, indent=2))
    if "per_class" in score:
        per_class_df = pd.DataFrame.from_dict(score["per_class"], orient="index")
        display(per_class_df)
else:
    VAL_SCORE = None
    print("Evaluator not found, skipped local validation scoring.")


Predict validation command:
/usr/bin/python3 predict.py --image_dir /kaggle/input/datasets/duong1589/object-detection/public/val/images --output /kaggle/working/object-detection/val_predictions.json --checkpoint /kaggle/working/object-detection/models/best.pth --batch_size 8 --conf_threshold 0.25 --nms_threshold 0.35 --max_detections 100
wrote 1500 predictions to /kaggle/working/object-detection/val_predictions.json
Validation predictions: images=1500 boxes=1979
Prediction preview:
[
  {
    "image_id": "img_00090df9158f.jpg",
    "boxes": [
      {
        "class": "dog",
        "confidence": 0.765793,
        "bbox": [
          115.04,
          12.6,
          500.0,
          362.4
        ]
      }
    ]
  },
  {
    "image_id": "img_003c3ddc6a82.jpg",
    "boxes": [
      {
        "class": "dog",
        "confidence": 0.615507,
        "bbox": [
          2.47,
          78.28,
          476.38,
          359.2
        ]
      }
    ]
  },
  {
    "image_id": "img_01073356e41c

,ap,num_ground_truth,num_predictions,true_positives,false_positives,recall,precision
person,0.721398,1074,1054,812,242,0.756052,0.770398
car,0.609924,283,294,184,110,0.650177,0.625850
dog,0.679803,206,221,153,68,0.742718,0.692308
cat,0.737607,176,165,134,31,0.761364,0.812121
chair,0.418734,282,245,140,105,0.496454,0.571429


## 7. Threshold Sweep On Validation


In [7]:
import json
import pandas as pd

BEST_CHECKPOINT = PROJECT_DIR / "models" / "best.pth"

if RUN_THRESHOLD_SWEEP:
    if not (VAL_JSON.exists() and EVALUATOR.exists()):
        raise FileNotFoundError("Need VAL_JSON and EVALUATOR for threshold sweep.")
    rows = []
    sweep_dir = PROJECT_DIR / "threshold_sweep"
    sweep_dir.mkdir(parents=True, exist_ok=True)
    for conf in CONF_VALUES:
        for nms in NMS_VALUES:
            sweep_pred = sweep_dir / f"predictions_conf{conf:.2f}_nms{nms:.2f}.json"
            sweep_score = sweep_dir / f"score_conf{conf:.2f}_nms{nms:.2f}.json"
            print(f"Running conf={conf:.2f} nms={nms:.2f}")
            !{sys.executable} predict.py --image_dir "{VAL_IMAGES}" --output "{sweep_pred}" --checkpoint "{BEST_CHECKPOINT}" --batch_size {BATCH_SIZE} --conf_threshold {conf} --nms_threshold {nms} --max_detections {MAX_DETECTIONS}
            !{sys.executable} "{EVALUATOR}" --ground_truth "{VAL_JSON}" --predictions "{sweep_pred}" --output "{sweep_score}"
            score = json.loads(sweep_score.read_text(encoding="utf-8"))
            rows.append({
                "conf": conf,
                "nms": nms,
                "map50": score.get("mAP@0.5", 0.0),
                "performance_points": score.get("performance_points", 0),
                "precision": score.get("micro_precision", 0.0),
                "recall": score.get("micro_recall", 0.0),
                "num_predictions": score.get("num_predictions", 0),
            })
    sweep_df = pd.DataFrame(rows).sort_values("map50", ascending=False)
    sweep_csv = sweep_dir / "threshold_sweep_results.csv"
    sweep_df.to_csv(sweep_csv, index=False)
    print("Threshold sweep results:", sweep_csv)
    display(sweep_df)
else:
    print("Threshold sweep disabled. Set RUN_THRESHOLD_SWEEP = True to run it.")


Running conf=0.05 nms=0.35
wrote 1500 predictions to /kaggle/working/object-detection/threshold_sweep/predictions_conf0.05_nms0.35.json
{
  "mAP@0.5": 0.685611,
  "performance_points": 20,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 7257,
  "micro_precision": 0.226678,
  "micro_recall": 0.813953,
  "per_class": {
    "person": {
      "ap": 0.771993,
      "num_ground_truth": 1074,
      "num_predictions": 3345,
      "true_positives": 908,
      "false_positives": 2437,
      "recall": 0.845438,
      "precision": 0.27145
    },
    "car": {
      "ap": 0.671339,
      "num_ground_truth": 283,
      "num_predictions": 1426,
      "true_positives": 226,
      "false_positives": 1200,
      "recall": 0.798587,
      "precision": 0.158485
    },
    "dog": {
      "ap": 0.725524,
      "num_ground_truth": 206,
      "num_predictions": 588,
      "true_positives": 171,
      "false_positives": 417,
      "recall": 0.830097,
      "precision": 0.290816
  

,conf,nms,map50,performance_points,precision,recall,num_predictions
1,0.05,0.45,0.688330,20,0.176013,0.840178,9647
0,0.05,0.35,0.685611,20,0.226678,0.813953,7257
2,0.05,0.50,0.679242,20,0.147730,0.848590,11609
5,0.10,0.45,0.674909,20,0.368928,0.796635,4364
4,0.10,0.35,0.672951,20,0.444980,0.776348,3526
6,0.10,0.50,0.667254,20,0.313054,0.804552,5194
9,0.15,0.45,0.661658,20,0.502779,0.761009,3059
8,0.15,0.35,0.660091,20,0.568250,0.745670,2652
10,0.15,0.50,0.653989,20,0.438171,0.767937,3542
12,0.20,0.35,0.646681,20,0.661371,0.720930,2203


## 8. Predict On Hidden/Test Images If Available


In [8]:
if TEST_IMAGE_DIR is None:
    print("TEST_IMAGE_DIR is None. Skipped final prediction.")
else:
    test_dir = Path(TEST_IMAGE_DIR)
    final_cmd = [
        sys.executable, "predict.py",
        "--image_dir", str(test_dir),
        "--output", str(FINAL_PREDICTIONS),
        "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
        "--batch_size", str(BATCH_SIZE),
        "--conf_threshold", str(CONF_THRESHOLD),
        "--nms_threshold", str(NMS_THRESHOLD),
    ]
    print("Final predict command:")
    print(" ".join(final_cmd))
    subprocess.run(final_cmd, check=True)
    final_data = json.loads(FINAL_PREDICTIONS.read_text(encoding="utf-8"))
    print(f"Final predictions: images={len(final_data)} boxes={sum(len(item['boxes']) for item in final_data)}")
    print("Final prediction preview:")
    print(json.dumps(final_data[:3], ensure_ascii=False, indent=2))
    print("Final predictions path:", FINAL_PREDICTIONS)


TEST_IMAGE_DIR is None. Skipped final prediction.


## 9. Collect Artifacts


In [9]:
import shutil

ARTIFACT_DIR = Path("/kaggle/working/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    PROJECT_DIR / "models" / "best.pth",
    PROJECT_DIR / "models" / "last.pth",
    VAL_PREDICTIONS,
    PROJECT_DIR / "val_score.json",
]
files_to_copy += sorted((PROJECT_DIR / "models").glob("train_history_*.csv"))
sweep_results = PROJECT_DIR / "threshold_sweep" / "threshold_sweep_results.csv"
if sweep_results.exists():
    files_to_copy.append(sweep_results)
if FINAL_PREDICTIONS.exists():
    files_to_copy.append(FINAL_PREDICTIONS)

copied = []
for src in files_to_copy:
    if src.exists():
        dst = ARTIFACT_DIR / src.name
        shutil.copy2(src, dst)
        copied.append(dst)

zip_path = shutil.make_archive("/kaggle/working/object_detection_artifacts", "zip", ARTIFACT_DIR)
print("Artifacts directory:", ARTIFACT_DIR)
print("Artifacts zip:", zip_path)
print("Files:")
artifact_rows = []
for path in sorted(ARTIFACT_DIR.iterdir()):
    artifact_rows.append({"file": path.name, "size_mb": round(path.stat().st_size / (1024 * 1024), 3)})
display(pd.DataFrame(artifact_rows))


Artifacts directory: /kaggle/working/artifacts
Artifacts zip: /kaggle/working/object_detection_artifacts.zip
Files:


,file,size_mb
0,best.pth,191.605
1,last.pth,191.605
2,threshold_sweep_results.csv,0.001
3,train_history_20260529_154823.csv,0.049
4,val_predictions.json,0.421
5,val_score.json,0.001


## 10. Summary Shown In Notebook


In [10]:
summary = {
    "best_checkpoint": str(PROJECT_DIR / "models" / "best.pth"),
    "history_csv": str(HISTORY_CSV) if HISTORY_CSV is not None else None,
    "val_predictions": str(VAL_PREDICTIONS),
    "val_score": str(PROJECT_DIR / "val_score.json"),
    "threshold_sweep_results": str(PROJECT_DIR / "threshold_sweep" / "threshold_sweep_results.csv"),
    "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

if HISTORY_CSV is not None:
    history_df = pd.read_csv(HISTORY_CSV)
    print("Final history row:")
    display(history_df.tail(1))

score_path = PROJECT_DIR / "val_score.json"
if score_path.exists():
    score = json.loads(score_path.read_text(encoding="utf-8"))
    print(f"mAP@0.5={score.get('mAP@0.5')} performance_points={score.get('performance_points')}")


{
  "best_checkpoint": "/kaggle/working/object-detection/models/best.pth",
  "history_csv": "/kaggle/working/object-detection/models/train_history_20260529_154823.csv",
  "val_predictions": "/kaggle/working/object-detection/val_predictions.json",
  "val_score": "/kaggle/working/object-detection/val_score.json",
  "threshold_sweep_results": "/kaggle/working/object-detection/threshold_sweep/threshold_sweep_results.csv",
  "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip"
}
Final history row:


,run_started_at,epoch,total_epochs,train_data,val_data,image_dir,val_image_dir,image_size,batch_size,num_workers,...,train_objectness_loss,train_class_loss,train_num_positive,val_loss,val_box_loss,val_objectness_loss,val_class_loss,val_num_positive,val_map50,best_map50
79,2026-05-29 15:48:23,80,80,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,512,8,2,...,0.033669,0.008345,100.921109,1.922301,0.297798,0.302321,0.261984,89.308511,0.625121,0.633244


mAP@0.5=0.633493 performance_points=20
